# FLUX Multi-Model Orchestrator

**Architecture:** Multi-model registry with lazy loading

| Model | Size | Role | Loading |
|-------|------|------|--------|
| Qwen2.5-1.5B-Instruct | ~3GB | Main voice, tool routing | Always |
| Qwen2.5-VL-3B | ~6GB | Vision/spatial | Lazy |
| Qwen2.5-Coder-1.5B | ~3GB | Code generation | Lazy |
| Whisper-small | ~500MB | Audio → text | Lazy |
| TTS (Bark) | ~500MB | Text → speech | Lazy |
| Embedding | ~100MB | Text → 432-dim waves | Always |

This notebook:
1. Checks if models already embedded in .flx
2. Loads Instruct + Embedding (always-on)
3. Creates lazy loaders for VL, Coder, Whisper, TTS
4. Tests tool-calling format compliance
5. Tests model routing
6. Saves multi-model config to .flx

**Benefits:**
- 5-10x faster for text tasks
- Each model specialized for its task
- ~3.5GB baseline VRAM (instruct + embedding)
- Models load on-demand

**Requirements:**
- GPU with 8GB+ VRAM (16GB for all)
- Flux-Apex-V1.flx checkpoint
- HF_TOKEN

## Section 1: Setup

In [ ]:
%%time
"""Cell 1: Environment Setup"""

import os
import sys
import gc
import json
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Optional

import torch
import torch.nn as nn

# Detect environment
if Path('/kaggle').exists():
    ROOT = Path('/kaggle/working/FLUX')
    ENVIRONMENT = 'kaggle'
elif Path('/content').exists():
    ROOT = Path('/content/FLUX')
    ENVIRONMENT = 'colab'
else:
    ROOT = Path('/Users/admin/Desktop/flux')
    ENVIRONMENT = 'local'

if not ROOT.exists():
    print(f'  Cloning FLUX repository...')
    os.system(f'git clone https://github.com/Unseengap/FLUX.git {ROOT}')
else:
    os.chdir(ROOT)
    os.system('git pull --ff-only 2>/dev/null')

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.system('pip install -q transformers accelerate huggingface_hub sentence-transformers 2>/dev/null')

from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=103)
log.separator("FLUX Multi-Model Orchestrator")

device = get_device()
print(f'  Environment: {ENVIRONMENT}')
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

print('  ✓ Environment ready')

In [ ]:
"""Cell 2: Load HF Token & FLUX Model"""

log.cell_start("Cell 2 — Load Tokens & Model")

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('  ✓ Colab secrets loaded')
    except:
        pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  ✓ Environment HF_TOKEN loaded')

if hf_token:
    hf_token = hf_token.strip()
    os.environ['HF_TOKEN'] = hf_token

CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)
APEX_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1.flx'

if not APEX_PATH.exists():
    print(f'  Downloading Flux-Apex-V1.flx...')
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=hf_token,
    )
    print(f'  ✓ Downloaded')

print(f'\n  Loading {APEX_PATH.name}...')
flux_model = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)

print(f'\n  FLUX Model:')
print(f'    Version: {flux_model.get("version", "unknown")}')
print(f'    Keys: {len(flux_model)}')

# Check what's already embedded
existing_models = []
if 'orchestration' in flux_model:
    orch = flux_model['orchestration']
    if 'models' in orch:
        existing_models = list(orch['models'].keys())
    elif 'primary_model' in orch:
        existing_models.append('instruct')  # v5.3 format
    if 'vision_model' in orch:
        existing_models.append('vision')

print(f'    Existing orchestration models: {existing_models if existing_models else "none"}')

log.cell_end("Cell 2 — Load Tokens & Model", "PASS")

## Section 2: Model Registry

In [ ]:
"""Cell 3: Define Model Registry"""

log.cell_start("Cell 3 — Model Registry")

MODEL_REGISTRY = {
    'instruct': {
        'id': 'Qwen/Qwen2.5-1.5B-Instruct',
        'role': 'main_voice',
        'tasks': ['tool_calling', 'text_reasoning', 'instruction_following'],
        'always_loaded': True,
        'vram_gb': 3.0,
        'loader': 'AutoModelForCausalLM',
    },
    'vision': {
        'id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'role': 'vision_only',
        'tasks': ['image_understanding', 'spatial_graphs', 'grid_visualization'],
        'always_loaded': False,
        'vram_gb': 6.0,
        'loader': 'Qwen2_5_VLForConditionalGeneration',
    },
    'coder': {
        'id': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
        'role': 'code_generation',
        'tasks': ['generate_transform', 'write_python', 'code_explanation'],
        'always_loaded': False,
        'vram_gb': 3.0,
        'loader': 'AutoModelForCausalLM',
    },
    'whisper': {
        'id': 'openai/whisper-small',
        'role': 'speech_to_text',
        'tasks': ['transcribe_audio', 'voice_input'],
        'always_loaded': False,
        'vram_gb': 0.5,
        'loader': 'WhisperForConditionalGeneration',
    },
    'tts': {
        'id': 'suno/bark-small',
        'role': 'text_to_speech',
        'tasks': ['speak', 'voice_output'],
        'always_loaded': False,
        'vram_gb': 0.5,
        'loader': 'BarkModel',
    },
    'embedding': {
        'id': 'sentence-transformers/all-MiniLM-L6-v2',
        'role': 'wave_encoder',
        'tasks': ['text_to_wave', 'semantic_similarity'],
        'always_loaded': True,
        'vram_gb': 0.1,
        'loader': 'SentenceTransformer',
    },
}

print(f'  ═══ MODEL REGISTRY ═══\n')
for name, spec in MODEL_REGISTRY.items():
    status = '✓' if name in existing_models else '○'
    load = 'always' if spec['always_loaded'] else 'lazy'
    print(f'  {status} {name}: {spec["id"]}')
    print(f'      VRAM: {spec["vram_gb"]}GB | Load: {load}')

total_always = sum(s['vram_gb'] for s in MODEL_REGISTRY.values() if s['always_loaded'])
total_all = sum(s['vram_gb'] for s in MODEL_REGISTRY.values())
print(f'\n  VRAM budget:')
print(f'    Always loaded: {total_always:.1f} GB')
print(f'    All models: {total_all:.1f} GB')

log.cell_end("Cell 3 — Model Registry", "PASS")

## Section 3: Load Always-On Models

In [ ]:
%%time
"""Cell 4: Load Instruct Model (Main Voice)"""

log.cell_start("Cell 4 — Load Instruct")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer

INSTRUCT_MODEL_ID = MODEL_REGISTRY['instruct']['id']

# Check if already in orchestration (skip embedding, just load for runtime)
skip_embed_instruct = 'instruct' in existing_models
if skip_embed_instruct:
    print(f'  ℹ Instruct already in .flx orchestration config')
    print(f'  Loading from HF for runtime use...\n')
else:
    print(f'  Loading {INSTRUCT_MODEL_ID}...\n')

try:
    instruct_tokenizer = AutoTokenizer.from_pretrained(
        INSTRUCT_MODEL_ID,
        trust_remote_code=True,
    )
    
    instruct_model = AutoModelForCausalLM.from_pretrained(
        INSTRUCT_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto" if device == 'cuda' else None,
        trust_remote_code=True,
    )
    instruct_model.eval()
    
    params = sum(p.numel() for p in instruct_model.parameters())
    print(f'  ✓ Instruct model loaded')
    print(f'    Parameters: {params:,} ({params/1e9:.2f}B)')
    print(f'    Device: {next(instruct_model.parameters()).device}')
    instruct_ready = True
    
except Exception as e:
    print(f'  ✗ Failed: {e}')
    instruct_ready = False

if device == 'cuda':
    print(f'    VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

log.cell_end("Cell 4 — Load Instruct", "PASS" if instruct_ready else "FAIL")

In [ ]:
"""Cell 5: Load Embedding Model"""

log.cell_start("Cell 5 — Load Embedding")

print(f'  Loading embedding model for wave conversion...')

try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(MODEL_REGISTRY['embedding']['id'])
    if device == 'cuda':
        embedding_model = embedding_model.to(device)
    
    # Test it
    test_embed = embedding_model.encode(["test"])
    print(f'  ✓ Embedding model loaded')
    print(f'    Output dim: {test_embed.shape[1]} (will project to 432)')
    embedding_ready = True
    
except ImportError:
    print(f'  ⚠ sentence-transformers not installed')
    embedding_model = None
    embedding_ready = False
except Exception as e:
    print(f'  ⚠ Failed: {e}')
    embedding_model = None
    embedding_ready = False

# Create projection layer (384 → 432)
if embedding_ready:
    embed_to_wave = nn.Linear(384, 432)
    if device == 'cuda':
        embed_to_wave = embed_to_wave.to(device)
    print(f'  ✓ Projection layer: 384 → 432')

log.cell_end("Cell 5 — Load Embedding", "PASS" if embedding_ready else "SKIP")

In [ ]:
"""Cell 6: Define Generation Functions"""

log.cell_start("Cell 6 — Generation Functions")

def generate_instruct(
    prompt: str,
    system_prompt: str = "",
    max_new_tokens: int = 256,
    temperature: float = 0.1,
    do_sample: bool = False,
) -> str:
    """Generate with instruct model."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    text = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = instruct_tokenizer(text, return_tensors="pt").to(instruct_model.device)
    
    with torch.no_grad():
        outputs = instruct_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if do_sample else None,
            do_sample=do_sample,
            pad_token_id=instruct_tokenizer.eos_token_id,
        )
    
    response = instruct_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

def text_to_wave(text: str) -> torch.Tensor:
    """Convert text to 432-dim wave using embedding model."""
    if not embedding_ready:
        return torch.randn(432)
    with torch.no_grad():
        embed = embedding_model.encode([text], convert_to_tensor=True)
        wave = embed_to_wave(embed.float().detach())
    return wave.squeeze(0)

# Quick test
if instruct_ready:
    test = generate_instruct("Say 'ready' and nothing else.")
    print(f'  Instruct test: "{test}"')

if embedding_ready:
    wave = text_to_wave("test pattern")
    print(f'  Embedding test: wave shape = {wave.shape}')

print(f'  ✓ Generation functions ready')

log.cell_end("Cell 6 — Generation Functions", "PASS")

## Section 4: Lazy Model Loaders

In [ ]:
"""Cell 7: Lazy Model Loaders"""

log.cell_start("Cell 7 — Lazy Loaders")

class LazyModel:
    """Base class for lazy-loaded models."""
    
    def __init__(self, name: str, spec: Dict[str, Any]):
        self.name = name
        self.spec = spec
        self._model = None
        self._processor = None
        self._loaded = False
    
    @property
    def is_loaded(self) -> bool:
        return self._loaded
    
    def _load(self):
        raise NotImplementedError


class LazyVLModel(LazyModel):
    """Lazy-loaded VL model for vision tasks."""
    
    def _load(self):
        if self._loaded:
            return
        print(f'  Loading VL model...')
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        
        self._processor = AutoProcessor.from_pretrained(
            self.spec['id'], trust_remote_code=True,
        )
        self._model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.spec['id'],
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
        
        # Load embedded weights if available
        if 'vlm' in flux_model and 'weights' in flux_model['vlm']:
            weights = flux_model['vlm']['weights']
            remapped = {}
            for k, v in weights.items():
                if k.startswith('model.'):
                    remapped['model.language_model.' + k[6:]] = v
                elif k.startswith('visual.'):
                    remapped['model.' + k] = v
                else:
                    remapped[k] = v
            self._model.load_state_dict(remapped, strict=False)
            print(f'  ✓ Loaded {len(weights)} embedded weights')
        
        self._model.eval()
        self._loaded = True
        print(f'  ✓ VL model ready')
    
    def generate(self, prompt: str, images: List = None, max_new_tokens: int = 256) -> str:
        self._load()
        messages = [{"role": "user", "content": prompt}]
        text = self._processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        if images:
            inputs = self._processor(text=[text], images=images, return_tensors="pt").to(self._model.device)
        else:
            inputs = self._processor(text=[text], return_tensors="pt").to(self._model.device)
        
        with torch.no_grad():
            outputs = self._model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        response = self._processor.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response.lower():
            response = response.split("assistant")[-1].strip()
        return response


class LazyCoderModel(LazyModel):
    """Lazy-loaded Coder model for code generation."""
    
    def _load(self):
        if self._loaded:
            return
        print(f'  Loading Coder model...')
        from transformers import AutoModelForCausalLM, AutoTokenizer
        
        self._processor = AutoTokenizer.from_pretrained(
            self.spec['id'], trust_remote_code=True,
        )
        self._model = AutoModelForCausalLM.from_pretrained(
            self.spec['id'],
            torch_dtype=torch.float16,
            device_map="auto" if device == 'cuda' else None,
            trust_remote_code=True,
        )
        self._model.eval()
        self._loaded = True
        print(f'  ✓ Coder model ready')
    
    def generate(self, prompt: str, max_new_tokens: int = 512) -> str:
        self._load()
        messages = [{"role": "user", "content": prompt}]
        text = self._processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self._processor(text, return_tensors="pt").to(self._model.device)
        
        with torch.no_grad():
            outputs = self._model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=self._processor.eos_token_id,
            )
        response = self._processor.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response.lower():
            response = response.split("assistant")[-1].strip()
        return response


class LazyWhisperModel(LazyModel):
    """Lazy-loaded Whisper model for speech-to-text."""
    
    def _load(self):
        if self._loaded:
            return
        print(f'  Loading Whisper model...')
        from transformers import WhisperForConditionalGeneration, WhisperProcessor
        
        self._processor = WhisperProcessor.from_pretrained(self.spec['id'])
        self._model = WhisperForConditionalGeneration.from_pretrained(
            self.spec['id'],
            torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        )
        if device == 'cuda':
            self._model = self._model.to(device)
        self._model.eval()
        self._loaded = True
        print(f'  ✓ Whisper model ready')
    
    def transcribe(self, audio_array, sampling_rate: int = 16000) -> str:
        self._load()
        inputs = self._processor(
            audio_array, sampling_rate=sampling_rate, return_tensors="pt"
        )
        if device == 'cuda':
            inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            generated_ids = self._model.generate(**inputs)
        return self._processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


class LazyTTSModel(LazyModel):
    """Lazy-loaded TTS model for text-to-speech."""
    
    def _load(self):
        if self._loaded:
            return
        print(f'  Loading TTS model...')
        from transformers import BarkModel, AutoProcessor
        
        self._processor = AutoProcessor.from_pretrained(self.spec['id'])
        self._model = BarkModel.from_pretrained(
            self.spec['id'],
            torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        )
        if device == 'cuda':
            self._model = self._model.to(device)
        self._model.eval()
        self._loaded = True
        print(f'  ✓ TTS model ready')
    
    def speak(self, text: str, voice_preset: str = "v2/en_speaker_6"):
        self._load()
        inputs = self._processor(text, voice_preset=voice_preset, return_tensors="pt")
        if device == 'cuda':
            inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            audio = self._model.generate(**inputs)
        return audio.cpu().numpy().squeeze()


# Create lazy loaders
lazy_models = {
    'vision': LazyVLModel('vision', MODEL_REGISTRY['vision']),
    'coder': LazyCoderModel('coder', MODEL_REGISTRY['coder']),
    'whisper': LazyWhisperModel('whisper', MODEL_REGISTRY['whisper']),
    'tts': LazyTTSModel('tts', MODEL_REGISTRY['tts']),
}

print(f'  Created {len(lazy_models)} lazy loaders:')
for name, loader in lazy_models.items():
    print(f'    • {name}: {MODEL_REGISTRY[name]["id"]} (loaded: {loader.is_loaded})')

print(f'\n  ✓ Lazy loaders ready (no VRAM used yet)')

log.cell_end("Cell 7 — Lazy Loaders", "PASS")

## Section 5: Tools & Routing

In [ ]:
"""Cell 8: Define Tools with Model Routing"""

log.cell_start("Cell 8 — Tools")

FLUX_TOOLS = {
    'encode_grid': {
        'description': 'Encode an ARC grid into a semantic wave',
        'model': None,  # Uses embedding
        'example': '<tool>encode_grid</tool><params>{"grid": [[0,1],[1,0]], "mode": "holistic"}</params>',
    },
    'query_field': {
        'description': 'Query the resonance field for related concepts',
        'model': None,  # Native FLUX
        'example': '<tool>query_field</tool><params>{"wave": "$LAST_WAVE", "top_k": 5}</params>',
    },
    'recall_memory': {
        'description': 'Recall episodic memories',
        'model': None,  # Native FLUX
        'example': '<tool>recall_memory</tool><params>{"query": "rotation", "memory_type": "episodic"}</params>',
    },
    'decode_grid': {
        'description': 'Decode a wave back into an ARC grid',
        'model': None,  # Native FLUX
        'example': '<tool>decode_grid</tool><params>{"wave": "$LAST_WAVE", "height": 3, "width": 3}</params>',
    },
    'compare_grids': {
        'description': 'Compare two grids and find transformation rules',
        'model': None,  # Native FLUX
        'example': '<tool>compare_grids</tool><params>{"input_grid": [[0,1],[1,0]], "output_grid": [[1,0],[0,1]]}</params>',
    },
    'generate_code': {
        'description': 'Generate Python code to apply a transformation',
        'model': 'coder',  # Uses Coder model
        'example': '<tool>generate_code</tool><params>{"task": "rotate grid 90 degrees", "language": "python"}</params>',
    },
    'analyze_image': {
        'description': 'Analyze an image or visual representation',
        'model': 'vision',  # Uses VL model
        'example': '<tool>analyze_image</tool><params>{"image": "$IMAGE", "question": "what patterns do you see?"}</params>',
    },
    'transcribe_audio': {
        'description': 'Convert speech to text',
        'model': 'whisper',  # Uses Whisper
        'example': '<tool>transcribe_audio</tool><params>{"audio": "$AUDIO"}</params>',
    },
    'speak': {
        'description': 'Convert text to speech',
        'model': 'tts',  # Uses TTS
        'example': '<tool>speak</tool><params>{"text": "Hello, I am FLUX."}</params>',
    },
}

# Routing triggers
MODEL_TRIGGERS = {
    'vision': ['image', 'picture', 'see', 'look', 'visual', 'graph', 'diagram', 'spatial'],
    'coder': ['code', 'python', 'function', 'implement', 'write', 'program', 'script'],
    'whisper': ['audio', 'voice', 'hear', 'listen', 'transcribe', 'speech'],
    'tts': ['speak', 'say', 'read aloud', 'voice output'],
}

print(f'  Defined {len(FLUX_TOOLS)} tools:')
for name, spec in FLUX_TOOLS.items():
    model = spec['model'] or 'native'
    print(f'    • {name} → {model}')

log.cell_end("Cell 8 — Tools", "PASS")

In [ ]:
"""Cell 9: Tool Calling System"""

log.cell_start("Cell 9 — Tool System")

TOOL_SYSTEM_PROMPT = """You are FLUX, a physics-inspired AI. You MUST respond with tool calls in XML format.

TOOLS:
- encode_grid: Encode a grid into a wave
- query_field: Query knowledge field  
- compare_grids: Compare input/output grids
- generate_code: Generate Python code
- recall_memory: Recall memories

FORMAT: <tool>name</tool><params>{"key": "value"}</params>

EXAMPLES:

User: Analyze this grid: [[0,1],[1,0]]
Assistant: <tool>encode_grid</tool><params>{"grid": [[0,1],[1,0]], "mode": "holistic"}</params>

User: Compare input [[1,0],[0,1]] with output [[0,1],[1,0]]
Assistant: <tool>compare_grids</tool><params>{"input_grid": [[1,0],[0,1]], "output_grid": [[0,1],[1,0]]}</params>

User: Find patterns similar to rotation
Assistant: <tool>query_field</tool><params>{"wave": "$LAST_WAVE", "top_k": 5}</params>

User: Write Python code to rotate a grid 90 degrees
Assistant: <tool>generate_code</tool><params>{"task": "rotate grid 90 degrees clockwise", "language": "python"}</params>

User: What do you remember about diagonal patterns?
Assistant: <tool>recall_memory</tool><params>{"query": "diagonal patterns", "memory_type": "episodic"}</params>

CRITICAL: Output ONLY the tool call. No text before or after. Start with <tool>.
"""

def parse_tool_calls(text: str) -> List[Dict[str, Any]]:
    """Extract tool calls from text."""
    pattern = r'<tool>\s*([^<]+?)\s*</tool>\s*<params>\s*([^<]+?)\s*</params>'
    matches = re.findall(pattern, text, re.DOTALL)
    calls = []
    for name, params_str in matches:
        try:
            params = json.loads(params_str.strip())
        except:
            params = {'raw': params_str.strip()}
        calls.append({'name': name.strip(), 'params': params})
    return calls

def has_tool_call(text: str) -> bool:
    return '<tool>' in text and '</tool>' in text

def detect_model_needed(text: str) -> Optional[str]:
    """Detect which lazy model might be needed."""
    text_lower = text.lower()
    for model, triggers in MODEL_TRIGGERS.items():
        if any(t in text_lower for t in triggers):
            return model
    return None

# Test
test = '<tool>encode_grid</tool><params>{"grid": [[0,1]], "mode": "holistic"}</params>'
parsed = parse_tool_calls(test)
print(f'  Parser test: {parsed}')
print(f'  Detect "write python code": {detect_model_needed("write python code")}')
print(f'  Detect "analyze this image": {detect_model_needed("analyze this image")}')
print(f'  ✓ Tool system ready')

log.cell_end("Cell 9 — Tool System", "PASS")

## Section 6: Test Tool Calling

In [ ]:
%%time
"""Cell 10: Test Tool Calling"""

log.cell_start("Cell 10 — Tool Tests")

print(f'\n  ═══ TOOL CALLING TESTS ═══\n')

test_cases = [
    ('Analyze this grid: [[0,1],[1,0]]', 'encode_grid'),
    ('Compare input [[1,0]] with output [[0,1]]', 'compare_grids'),
    ('Find similar patterns to diagonal symmetry', 'query_field'),
    ('Write Python code to rotate a grid 90 degrees', 'generate_code'),
]

results = []
for prompt, expected in test_cases:
    print(f'  Test: "{prompt[:50]}..."')
    
    response = generate_instruct(
        prompt=prompt,
        system_prompt=TOOL_SYSTEM_PROMPT,
        max_new_tokens=150,
    )
    
    has_tool = has_tool_call(response)
    tool_calls = parse_tool_calls(response) if has_tool else []
    correct = any(expected == c['name'] for c in tool_calls)
    
    status = '✓' if correct else ('○' if has_tool else '✗')
    print(f'    {status} Expected: {expected}, Got: {tool_calls[0]["name"] if tool_calls else "none"}')
    results.append(correct)

passed = sum(results)
print(f'\n  Results: {passed}/{len(results)} correct')
tool_calling_ok = passed >= len(results) // 2

log.cell_end("Cell 10 — Tool Tests", "PASS" if tool_calling_ok else "FAIL")

In [ ]:
"""Cell 11: Test Model Routing"""

log.cell_start("Cell 11 — Routing Tests")

print(f'\n  ═══ MODEL ROUTING TESTS ═══\n')

routing_tests = [
    ("What do you see in this image?", 'vision'),
    ("Write Python code to flip a matrix", 'coder'),
    ("Transcribe this audio clip", 'whisper'),
    ("Speak this text aloud", 'tts'),
    ("Analyze this grid: [[0,1]]", None),  # Native, no lazy model
]

correct = 0
for prompt, expected in routing_tests:
    detected = detect_model_needed(prompt)
    status = '✓' if detected == expected else '✗'
    print(f'  {status} "{prompt[:40]}..."')
    print(f'      Expected: {expected}, Detected: {detected}')
    if detected == expected:
        correct += 1

print(f'\n  Routing accuracy: {correct}/{len(routing_tests)}')

# Verify no lazy models loaded yet
loaded = [n for n, m in lazy_models.items() if m.is_loaded]
print(f'  Lazy models loaded: {loaded if loaded else "none (good!)"}')

routing_ok = correct >= len(routing_tests) - 1

log.cell_end("Cell 11 — Routing Tests", "PASS" if routing_ok else "FAIL")

## Section 7: Save Config

In [ ]:
"""Cell 12: Save Multi-Model Config"""

log.cell_start("Cell 12 — Save Config")

print(f'\n  ═══ SAVE MULTI-MODEL CONFIGURATION ═══\n')

# Build models config (only add new ones)
models_config = {}

for name, spec in MODEL_REGISTRY.items():
    if name in existing_models:
        print(f'  ○ {name}: already in .flx, skipping')
        continue
    
    models_config[name] = {
        'model_id': spec['id'],
        'role': spec['role'],
        'tasks': spec['tasks'],
        'always_loaded': spec['always_loaded'],
        'vram_gb': spec['vram_gb'],
    }
    print(f'  ✓ {name}: adding to config')

# Update orchestration
if 'orchestration' not in flux_model:
    flux_model['orchestration'] = {}

flux_model['orchestration']['architecture'] = 'multi_model'
flux_model['orchestration']['models'] = {**flux_model['orchestration'].get('models', {}), **models_config}
flux_model['orchestration']['tools'] = FLUX_TOOLS
flux_model['orchestration']['model_triggers'] = MODEL_TRIGGERS
flux_model['orchestration']['system_prompt'] = TOOL_SYSTEM_PROMPT

# Update version
old_version = flux_model.get('version', 'unknown')
flux_model['version'] = '5.4-multi-model'
flux_model['phase'] = 'phase_multi_model'

# Update metadata
if 'metadata' not in flux_model:
    flux_model['metadata'] = {}
flux_model['metadata']['last_modified'] = datetime.now().isoformat()
flux_model['metadata']['modified_components'] = ['orchestration']

caps = flux_model['metadata'].get('capabilities', [])
for cap in ['multi_model', 'coder', 'whisper', 'tts', 'embedding']:
    if cap not in caps:
        caps.append(cap)
flux_model['metadata']['capabilities'] = caps

print(f'\n  Updated config:')
print(f'    Version: {old_version} → 5.4-multi-model')
print(f'    Models: {len(flux_model["orchestration"]["models"])}')
print(f'    Tools: {len(FLUX_TOOLS)}')

config_ok = True

log.cell_end("Cell 12 — Save Config", "PASS")

In [ ]:
"""Cell 13: Save to Disk & Upload"""

log.cell_start("Cell 13 — Save & Upload")

SAVE_TO_DISK = True
UPLOAD_TO_HF = True

print(f'\n  ═══ SAVE & UPLOAD ═══\n')

if SAVE_TO_DISK:
    OUTPUT_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1.flx'
    print(f'  Saving to: {OUTPUT_PATH}')
    
    torch.save(flux_model, str(OUTPUT_PATH), pickle_protocol=4)
    
    size_gb = OUTPUT_PATH.stat().st_size / 1e9
    print(f'  ✓ Saved: {size_gb:.2f} GB')
    save_ok = True
else:
    save_ok = False

if UPLOAD_TO_HF and save_ok:
    try:
        from flux_utils import upload_flx_to_hf, _resolve_hf_token
        token = _resolve_hf_token(hf_token)
        if token:
            print(f'\n  Uploading to HuggingFace Hub...')
            upload_flx_to_hf(str(OUTPUT_PATH), hf_token=token)
            print(f'  ✓ Upload complete')
            upload_ok = True
        else:
            print(f'  ⚠ No HF token')
            upload_ok = False
    except Exception as e:
        print(f'  ✗ Upload failed: {e}')
        upload_ok = False
else:
    upload_ok = False

log.cell_end("Cell 13 — Save & Upload", "PASS" if save_ok else "SKIP")

## Section 8: Summary

In [ ]:
"""Cell 14: Final Summary"""

log.cell_start("Cell 14 — Summary")

print(f'\n  ═══════════════════════════════════════════════════════')
print(f'  ═══ FLUX MULTI-MODEL ORCHESTRATOR COMPLETE ═══')
print(f'  ═══════════════════════════════════════════════════════')

print(f'\n  Model Registry:')
for name, spec in MODEL_REGISTRY.items():
    if name == 'instruct':
        status = '✓ loaded' if instruct_ready else '○'
    elif name == 'embedding':
        status = '✓ loaded' if embedding_ready else '○'
    else:
        lm = lazy_models.get(name)
        status = '✓ loaded' if (lm and lm.is_loaded) else 'lazy'
    load = 'always' if spec['always_loaded'] else 'lazy'
    print(f'    {name}: {spec["vram_gb"]}GB, {load}, {status}')

print(f'\n  Test Results:')
results_table = [
    ('Instruct model', instruct_ready),
    ('Embedding model', embedding_ready),
    ('Tool calling', tool_calling_ok if 'tool_calling_ok' in dir() else False),
    ('Model routing', routing_ok if 'routing_ok' in dir() else False),
    ('Config saved', config_ok if 'config_ok' in dir() else False),
    ('File saved', save_ok if 'save_ok' in dir() else False),
]

passed = sum(1 for _, ok in results_table if ok)
for name, ok in results_table:
    print(f'    {"✓" if ok else "○"} {name}')

print(f'\n  Results: {passed}/{len(results_table)} passed')

print(f'\n  Model Info:')
print(f'    Version: {flux_model.get("version", "unknown")}')
print(f'    Models: {len(MODEL_REGISTRY)}')
print(f'    Tools: {len(FLUX_TOOLS)}')

if device == 'cuda':
    print(f'    VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

print(f'\n  ═══════════════════════════════════════════════════════')
print(f'  ✓ Multi-model architecture ready')
print(f'  ═══════════════════════════════════════════════════════')

log.cell_end("Cell 14 — Summary", "PASS")